In [ ]:
# pip install pandas
# pip install geopandas
# pip install matplotlib
# pip install folium

In [ ]:
#### Importing the required libraries

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
#### Extracting the required columns from the raw data file


cols = [
    'Year',
    'District',
    'Block',
    'Satellite',
    'Date',
    'Time (IST)',
    'Day / Night',
    'Fire Power(W/m2)',
    'Corrected_long',
    'corrected_lat',
    'Graama'
]

df = pd.read_csv(r'C:\Users\imvaish\Desktop\Stubble_Burning_Analysis\data\Punjab_Stubble_data_2018_21_Raw.csv', usecols=cols)
df.head()

In [ ]:
#### Read the data file and check the data types of each column

df.info()

In [ ]:
#### Convert the 'Date' column to datetime format, handling errors and 
####specifying that the day comes first in the date format.

df['Date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)

In [ ]:
#### Date column may have some invalid date formats 
# that could not be converted to datetime.

df[df['Date'].isna()].head(20)


In [ ]:
#### Display the rows where the 'Date' column is NaN 
# to identify any issues with the date format.

df.loc[df['Date'].isna(), 'Date']

In [ ]:
#### Check for missing values in the DataFrame 
# to identify any other potential data quality issues.

df.isnull().sum()

In [ ]:
#### Drop duplicate rows from the DataFrame 
# to ensure that each record is unique.

df = df.drop_duplicates()

In [ ]:
#### Filter the DataFrame to remove rows 
# where 'Corrected_long' is equal to 'corrected_lat',

df = df[df['Corrected_long'] != df['corrected_lat']]

In [ ]:
#### Filter the DataFrame to include only records falling within Punjab extent

df = df[
    df['Corrected_long'].between(73.5, 76.9) &
    df['corrected_lat'].between(29.5, 32.5)
]

In [ ]:
#### Extract month, month name, and year from the 'Date' column

df['month'] = df['Date'].dt.month
df['month_name'] = df['Date'].dt.month_name()
df['year'] = df['Date'].dt.year

In [ ]:
#### Display the count of records for each month name 
# to understand the distribution of data across different months.

df['month_name'].value_counts()

In [ ]:
#### Filter the DataFrame to include only records from September, October, and November

df_season = df[df['month_name'].isin(['September','October','November'])]

In [ ]:
#### Display the count of records for each district in the filtered DataFrame

df_season['District'].value_counts().head(10)

In [ ]:
##### Visualize the top 10 districts with the highest number of stubble burning records

top = df_season['District'].value_counts().head(10)

plt.figure(figsize=(10,5))
plt.bar(top.index, top.values)
plt.xticks(rotation=45)
plt.title("Top Districts with Stubble Burning")
plt.show()

In [ ]:
#### Calculate the average Fire Power for each month in the filtered DataFrame

df_season.groupby('month_name')['Fire Power(W/m2)'].mean()

In [ ]:
#### Visualize the average Fire Power for each month in the filtered DataFrame

df_season.groupby('month_name')['Fire Power(W/m2)'].mean().plot(kind='bar')
plt.title("Average Fire Intensity by Month")
plt.show()

In [ ]:
#### Save the cleaned and filtered DataFrame to a new CSV file for further analysis or sharing.

df_season.to_csv("stubble_burning_seasonal_cleaned.csv", index=False)